# 03. Equilibrium and Kinetic Profiles

Session 02 ended with a plasma that had broken down and was carrying current. This session asks
what it settled into, and how anyone knows: an equilibrium code never sees the plasma, only
magnetic probes, flux loops, coil currents and — when they exist — a handful of Thomson and
spectroscopy channels. We follow the chain from those measurements to a reconstructed
equilibrium, to flux coordinates, to a kinetic plasma state, and we keep asking how well each
step explains what was measured.

## Session Overview

> Once a discharge has formed, how do we represent its plasma state, and how do experimental
> measurements constrain that representation?

```text
experiment
    ↓
diagnostic measurements
    ↓
constraints + uncertainties
    ↓
equilibrium reconstruction
    ↓
flux coordinates
    ↓
kinetic profile reconstruction
    ↓
derived plasma state
```

By the end of this session you will be able to:

- state the assumptions behind the Grad-Shafranov equation, and name where they fail;
- distinguish 2-D fields, 1-D flux functions, boundaries, global parameters and radial
  coordinates — and say which of them an EFIT file stores and which VAFT derives;
- tell a shape *parameterisation* (Miller) from a force-balanced *solution* (Solov'ev), and a
  limited plasma from a single- or double-null one;
- place CHEASE, TES, TokaMaker and EFIT on the forward/reconstruction and
  fixed/free-boundary axes;
- read an EFIT fit channel by channel — measured, uncertainty, weight, reconstructed,
  residual — and notice when the weights, not the diagnostics, set the fit;
- map Thomson and charge-exchange channels into flux coordinates, fit profiles through them,
  and judge the fit by $\chi^2_\nu$ rather than by eye;
- keep measured, reconstructed, assumed and derived quantities apart;
- compare several times of one discharge, and several discharges, on one coordinate.

| shot | what it brings | where |
| --- | --- | --- |
| `SHOT` = 39915 | nine EFIT slices with their fitted constraints, and 66 fast-camera frames | Part I |
| 48224 | Thomson scattering, charge-exchange spectroscopy, fitted `core_profiles`, and three equilibria of one instant: magnetic EFIT, kinetic EFIT, CHEASE | Part I, steps 4, 7, 8 |
| 41524, 41672 | two more EFIT discharges with their constraints | Part II |

**Part I** (Guided Analysis) builds the vocabulary and then works through `SHOT`; **Part II**
(Integrated Analysis) repeats the equilibrium analysis on several times and several discharges.
Everything runs offline; every cell that reads `SHOT` is written so the same code runs on any
shot loaded from the database.

## Physical Context

### From force balance to one scalar equation

A plasma in equilibrium is not accelerating, so the magnetic force balances the pressure
gradient everywhere at once:

$$\nabla p = \mathbf{J}\times\mathbf{B}, \qquad \nabla\times\mathbf{B} = \mu_0\mathbf{J}, \qquad \nabla\cdot\mathbf{B} = 0 .$$

In an axisymmetric torus the field can be written with the poloidal flux $\psi(R,Z)$ and the
toroidal-field function $F = R B_\phi$,

$$\mathbf{B} = \nabla\psi\times\nabla\phi + F\,\nabla\phi ,$$

and force balance then says that $p$ and $F$ are constant on surfaces of constant $\psi$ and
collapses to the **Grad-Shafranov equation**

$$\Delta^* \psi \;=\; -\mu_0 R^2 \, p'(\psi) \;-\; F F'(\psi),
\qquad
\Delta^* \equiv R\frac{\partial}{\partial R}\!\left(\frac{1}{R}\frac{\partial}{\partial R}\right) + \frac{\partial^2}{\partial Z^2}.$$

Everything follows from $\psi$: the flux surfaces are its contours, the poloidal field is its
gradient, and the two free functions $p'(\psi)$ and $FF'(\psi)$ carry the pressure and the
current.

### What that equation assumes

- **Axisymmetry** — nothing depends on the toroidal angle $\phi$.
- **A static equilibrium** — no flow large enough for $\rho\,\mathbf{v}\cdot\nabla\mathbf{v}$
  to matter, and a plasma that changes slowly compared with the Alfvén time.
- **Isotropic scalar pressure** — $p_\parallel = p_\perp$.
- **Ideal-MHD force balance** — nested flux surfaces, no islands.

Each one fails somewhere. Strong toroidal rotation shifts the pressure outward on a surface;
fast-ion or heating anisotropy makes $p_\parallel \ne p_\perp$; error fields and RMPs break
axisymmetry; tearing modes grow magnetic islands and, when they overlap, a stochastic region
where "flux surface" has no meaning. Those are session 05's subject. Everything here assumes
the standard form, and it is worth saying so every time a number is quoted.

### Reconstruction against forward modelling

Those are two different jobs.

**EFIT reconstructs.** It is given magnetic measurements — flux loops, field probes, coil
currents, the plasma current — and finds the $\psi$, $p'$ and $FF'$ most consistent with them.
It answers *what was the plasma doing*, and its answer is only as good as the diagnostics
constrain it.

**CHEASE forward-models.** It is given a boundary and the two profiles, and solves the equation
on a fine mesh to high accuracy. It answers *what equilibrium do these inputs imply*. It cannot
tell you about a discharge it was not given.

They also differ in what is free. EFIT solves a **free-boundary** problem — the boundary is part
of the answer, and its grid extends past the plasma to the coils. CHEASE solves a
**fixed-boundary** problem — you hand it the boundary and it works only inside. That difference
matters in step 8, when we measure both.

## Load / Prepare Data

### The discharge

`SHOT` is the only place a shot number is written in Part I. The packaged sample runs offline;
the commented line below it is the lab-mode path, which loads any VEST discharge from the
database (it needs HSDS credentials).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import vaft
from vaft.omas.efit_quality import constraint_table, fit_quality_metrics, sigma_unit_factor

In [ ]:
SHOT = 39915
ods = vaft.omas.sample_ods(SHOT)
# ods = vaft.database.load(SHOT)  # lab mode: any shot, needs HSDS credentials

times = np.asarray(ods["equilibrium.time"], dtype=float)
print(f"shot {SHOT}: {', '.join(sorted(ods.keys()))}")
print(f"{len(times)} equilibrium slices, {times.min() * 1e3:.1f} - {times.max() * 1e3:.1f} ms")

### Which slices are equilibria?

A stored slice is not automatically a usable equilibrium. EFIT writes a slice for every time it
was asked about, including the ones where the current has already gone. Decide from the data:
a slice is **usable** when its plasma current is finite and non-zero *and* it has a boundary.
The **representative** slice is the usable one nearest the current peak; every later cell of
Part I uses `i_rep`, never a typed index.

In [ ]:
ip = np.full(len(times), np.nan)
n_boundary = np.zeros(len(times), dtype=int)
for i in range(len(times)):
    root = f"equilibrium.time_slice.{i}"
    if f"{root}.global_quantities.ip" in ods:
        ip[i] = float(ods[f"{root}.global_quantities.ip"])
    if f"{root}.boundary.outline.r" in ods:
        n_boundary[i] = len(ods[f"{root}.boundary.outline.r"])

usable = np.isfinite(ip) & (ip != 0) & (n_boundary > 2)
usable_slices = np.flatnonzero(usable)
i_rep = int(usable_slices[np.argmax(np.abs(ip[usable]))])
t_rep = float(times[i_rep])

print(" slice   time [ms]   Ip [kA]   boundary points   usable")
for i in range(len(times)):
    print(f"{i:6d}   {times[i] * 1e3:9.1f}   {ip[i] / 1e3:7.1f}   {n_boundary[i]:15d}   {usable[i]}")
print(f"usable slices: {len(usable_slices)} of {len(times)}")
print(f"representative slice: {i_rep} at {t_rep * 1e3:.1f} ms (the usable slice nearest the Ip peak)")

In [ ]:
unusable = np.flatnonzero(~usable)
if unusable.size:
    root = f"equilibrium.time_slice.{int(unusable[-1])}"
    print(f"slice {int(unusable[-1])} at {times[unusable[-1]] * 1e3:.1f} ms:")
    print("  ip         :", float(ods[f"{root}.global_quantities.ip"]))
    print("  psi_axis   :", float(ods[f"{root}.global_quantities.psi_axis"]))
    print("  psi_bound. :", float(ods[f"{root}.global_quantities.psi_boundary"]))
    print("  boundary   :", f"{root}.boundary.outline.r" in ods)
else:
    print("every stored slice is usable")

On 39915 the last slice is degenerate: the current is zero, there is no boundary, and
`psi_axis == psi_boundary`. It is real EFIT output, not a bug — and a reminder to *find* the
usable window from the data rather than index into it.

### The kinetic shot

39915 was reconstructed from magnetics alone. Shot 48224 additionally has **Thomson scattering**
(electron temperature and density) and **charge-exchange spectroscopy** (C$^{3+}$ ion temperature
and toroidal velocity), the `core_profiles` fitted through them, and three equilibria of the same
instant: an EFIT reconstruction without the kinetic pressure constraint, the kinetic EFIT that
used it, and the CHEASE refinement of the kinetic one. It lives in the repository rather than the
installed package, so this cell needs a clone.

In [ ]:
KINETIC_SHOT = 48224
kinetic = vaft.omas.sample_ods(KINETIC_SHOT)
equilibria = vaft.omas.sample_equilibria(KINETIC_SHOT)
eq_magnetic = equilibria["efit_magnetic"]
eq_kinetic = equilibria["efit_kinetic"]
eq_chease = equilibria["chease"]

t_kinetic = float(np.asarray(kinetic["equilibrium.time"], dtype=float).min())
print(f"shot {KINETIC_SHOT}: {', '.join(sorted(kinetic.keys()))}")
print(f"Thomson: {len(kinetic['thomson_scattering.channel'])} channels, "
      f"charge exchange: {len(kinetic['charge_exchange.channel'])} channels, "
      f"equilibrium at {t_kinetic * 1e3:.0f} ms")
print("equilibria:", ", ".join(equilibria))

## Guided Analysis

### Step 1 — how an equilibrium is represented

An equilibrium is a hierarchy, not a boundary curve.

| level | quantities | what it is for |
| --- | --- | --- |
| **2-D** | $\psi(R,Z)$; $B_R, B_Z, B_\phi$; the LCFS; the magnetic axis; X-points; the limiter/wall | geometry, field-line tracing, mapping a diagnostic's $(R,Z)$ |
| **1-D flux functions** | $q(\psi)$, $p(\psi)$, $F(\psi) = R B_\phi$, $p'(\psi)$, $FF'(\psi)$, $j_\phi$ | the physics the solver actually solved for |
| **global** | $I_p$, $R_0$, $a$, $A = R_0/a$, $\kappa$, $\delta$, $q_{95}$, $\beta_p$, $\beta_N$, $\ell_i$ | comparing discharges and machines |
| **radial coordinates** | $\psi_N$, $\rho_\mathrm{pol} = \sqrt{\psi_N}$, $\rho_\mathrm{tor}$, $\rho_{\mathrm{tor},N}$, minor radius, outboard-midplane $R$ | the abscissa of every profile |

Start with the 2-D level on the representative slice: the flux map with the wall, the last
closed flux surface (LCFS), the magnetic axis and any X-point on it; then three fields computed
from it — the toroidal current density, the vertical field and the toroidal field. Only $\psi$ is
stored; `field=` derives the others on the fly without writing them into `ods`.

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(18, 6))
for axis, field in zip(axes, ("psi", "j_tor", "b_field_z", "b_field_tor")):
    vaft.omas.plot_equilibrium_field_2d(ods, time_slice=i_rep, field=field, ax=axis,
                                        overlay=("wall", "boundary", "axis", "x_points"))
figure.tight_layout()
plt.show()

The 1-D level: `q`, `pressure`, `p'` and `FF'` are the equilibrium's own description of itself —
the two source functions EFIT fitted, and what they integrate to.

In [ ]:
vaft.omas.plot_equilibrium_overview_profiles(ods, time_slice=i_rep)
plt.show()

The global level, as stored. Some of the quantities in the table above are simply not there
yet — step 5 derives them.

In [ ]:
stored = ods[f"equilibrium.time_slice.{i_rep}.global_quantities"]
for name in ("ip", "magnetic_axis.r", "magnetic_axis.z", "psi_axis", "psi_boundary",
             "q_axis", "q_95", "beta_pol", "beta_normal", "li_3"):
    print(f"{name:16s}: {float(stored[name]):12.5g}" if name in stored else f"{name:16s}: not stored")

And the radial coordinates. VAFT's `coordinate=` option selects between them:

| coordinate | is | useful for |
| --- | --- | --- |
| `psi_norm` | normalised poloidal flux $\psi_N$ | anything read straight off the flux map |
| `rho_tor_norm` | the stored `profiles_1d.rho_tor_norm` leaf, by definition $\sqrt{\Phi/\Phi_b}$ — on these samples a $\sqrt{\psi_N}$ proxy, which VAFT detects and replaces by one derived from $q$ | transport, where volume matters |
| `sqrt_phi_norm` | $\sqrt{\Phi/\Phi_b}$ computed from the toroidal flux itself: the stored `profiles_1d.phi`, else $\Phi = \int q\,d\psi$ | checking the stored leaf against its definition |
| `r_major` | outboard-midplane major radius | comparing with a midplane diagnostic |
| `r_minor` | minor radius | geometric reasoning |

$\rho_{\mathrm{tor},N}$ needs the toroidal flux $\Phi = \int q\,d\psi$, so it is not
$\sqrt{\psi_N}$ unless $q$ is flat. Check what the file carries before trusting it:

In [ ]:
profiles = ods[f"equilibrium.time_slice.{i_rep}.profiles_1d"]
psi_n = (np.asarray(profiles["psi"]) - float(stored["psi_axis"])) / (
    float(stored["psi_boundary"]) - float(stored["psi_axis"]))
if "rho_tor_norm" in profiles:
    gap = float(np.nanmax(np.abs(np.asarray(profiles["rho_tor_norm"]) - np.sqrt(psi_n))))
    print(f"stored rho_tor_norm vs sqrt(psi_N): max difference {gap:.2e}")
    print("stored rho_tor_norm is sqrt(psi_N):", gap < 1e-6)
else:
    print("no rho_tor_norm stored on this slice")

On the packaged 39915 slices the stored `rho_tor_norm` is exactly $\sqrt{\psi_N}$ — a proxy
written by the processing chain, not the toroidal-flux radius. Step 5 replaces it with the real
one, and step 6 shows how different the two are.

### Step 2 — analytic geometry and equilibria

#### Miller: a shape, not a solution

The Miller parameterisation describes one flux surface by five numbers — centre $R_0, Z_0$,
minor radius $r$, elongation $\kappa$ and triangularity $\delta$:

$$R(\theta) = R_0 + r\cos\!\big(\theta + \arcsin\delta\,\sin\theta\big), \qquad Z(\theta) = Z_0 + \kappa\, r \sin\theta .$$

Start from this discharge's own boundary: fit a Miller surface to the representative LCFS, then
scan around it.

In [ ]:
lcfs = ods[f"equilibrium.time_slice.{i_rep}.boundary.outline"]
lcfs_r, lcfs_z = np.asarray(lcfs["r"], dtype=float), np.asarray(lcfs["z"], dtype=float)
miller_fit = vaft.process.equilibrium.fit_miller_surface((lcfs_r, lcfs_z))
vest = miller_fit.surface
print(f"Miller fit to the LCFS of slice {i_rep}: R0 = {vest.r0:.3f} m, a = {vest.r:.3f} m, "
      f"A = {vest.r0 / vest.r:.2f}, kappa = {vest.kappa:.2f}, delta = {vest.delta:.2f}")
print(f"rms misfit / a = {miller_fit.normalized_rms_error:.3f}, accepted: {miller_fit.accepted}")

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(16, 5))
vaft.plot.plot_miller_surfaces(vest, ax=axes[0], title=f"#{SHOT} LCFS and its Miller fit")
axes[0].plot(lcfs_r, lcfs_z, "k--", lw=1, label="EFIT LCFS")
axes[0].legend(fontsize="small")
vaft.plot.plot_miller_surfaces(
    vaft.process.equilibrium.miller_surfaces(vest.r, r0=vest.r * np.array([1.3, 2.0, 3.0, 5.0]),
                                             kappa=vest.kappa, delta=vest.delta),
    ax=axes[1], title="aspect ratio",
)
vaft.plot.plot_miller_surfaces(
    vaft.process.equilibrium.miller_surfaces(vest.r, r0=vest.r0, kappa=[1.0, 1.5, 2.0, 2.5]),
    ax=axes[2], title="circular to elongated",
)
vaft.plot.plot_miller_surfaces(
    vaft.process.equilibrium.miller_surfaces(vest.r, r0=vest.r0, kappa=vest.kappa,
                                             delta=np.linspace(-0.6, 0.6, 5)),
    ax=axes[3], title="negative to positive triangularity",
)
figure.tight_layout()
plt.show()

VEST is a low-aspect-ratio machine — the first panel prints $A$ — so its inboard side is pressed
against the centre stack, where the field is strongest. The scans show what each parameter does
to the *outline*, and nothing else: a Miller surface carries no $p'$ and no $FF'$, and a family
of them nested at will need not satisfy any force balance. Miller is how gyrokinetic and
stability codes describe a local surface *given* an equilibrium; it is not an equilibrium.

#### Solov'ev: an exact solution

Take $p'$ and $FF'$ constant. The Grad-Shafranov equation then has closed-form solutions
(Solov'ev; the Cerfon-Freidberg basis adds enough homogeneous terms to impose a prescribed
boundary and an X-point on it). `solovev_example` builds one for each boundary topology, and
`derive_boundary_representation` classifies the result from the flux map alone — saddles of
$\psi$ are X-points only when their flux matches the boundary's.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 6))
for axis, topology in zip(axes, ("limited", "single_null", "double_null")):
    analytic = vaft.process.equilibrium.solovev_example(topology)
    vaft.plot.plot_solovev_equilibrium(analytic, ax=axis)
    boundary = vaft.process.equilibrium.derive_boundary_representation(analytic)
    active = [point for point in boundary.x_points if point.active]
    print(f"{topology:12s} -> topology {boundary.topology.value:18s} active X-points: "
          + (", ".join(f"(R={p.r:.3f}, Z={p.z:+.3f})" for p in active) or "none"))
figure.tight_layout()
plt.show()

```text
shape parameterisation   (Miller: five numbers per surface, any nesting)
        ≠
force-balanced solution  (Solov'ev: every surface follows from p', FF' and one boundary)
```

The topologies are a statement about the flux map, not about the outline. A **limited** plasma's
LCFS is the last surface that clears the wall — it touches the limiter and no X-point is on it.
A **diverted** plasma's LCFS is the separatrix through a saddle of $\psi$ — one (single null) or
two (double null) — and field lines outside it are led to strike points instead of to the wall.
VEST discharges are mostly limited on the centre stack, so X-points on real VEST maps are usually
saddles of the vacuum field *outside* the plasma, not on its boundary. The same classifier is run
on `SHOT` in step 5.

### Step 3 — which numerical problem are we solving?

Two independent axes. **Forward** solvers are given the boundary or the coil currents plus
$p'$ and $FF'$ and return $\psi$; **reconstruction** solvers are given measurements and return
the $\psi$, $p'$ and $FF'$ that best explain them. **Fixed-boundary** solvers work inside a
prescribed LCFS; **free-boundary** solvers include the coils and let the plasma find its edge.

| | fixed boundary | free boundary |
| --- | --- | --- |
| **forward** | CHEASE (`vaft.code.chease`) | TES (`vaft.code.tes`), TokaMaker (`vaft.code.tokamaker`) |
| **reconstruction** | not in VAFT (a fixed-boundary inverse fit to profiles) | EFIT, magnetic and kinetic (`vaft.code.efit`) |

EFIT is the only reconstruction VAFT runs, and the only one this session reads. CHEASE appears
twice below: as the fixed-boundary refinement of a kinetic EFIT (step 8) and as the forward model
of the exercise.

### Step 4 — what does the experiment actually constrain?

An equilibrium code does not observe the plasma. What it knows is a list of numbers, each with an
uncertainty, turned into a constraint:

$$\text{measurement} \rightarrow \text{uncertainty} \rightarrow \text{constraint} \rightarrow \text{model prediction} \rightarrow \text{residual}$$

#### 4.1 The magnetic inputs of VEST EFIT

| input | meaning | role | kind |
| --- | --- | --- | --- |
| plasma current $I_p$ | total toroidal current (Rogowski) | total-current constraint | measured |
| PF coil currents | known external sources | vacuum field | actuator (fitted to their measurement) |
| B-pol probes | local poloidal field at the wall | boundary shape, current distribution | measured |
| flux loops | poloidal flux at fixed points | flux geometry, boundary | measured |
| diamagnetic flux | toroidal flux the plasma expels | pressure-sensitive integral | measured |
| vessel eddy currents | induced wall currents | external magnetic response | modelled (session 02's solve) |
| limiter / wall | machine geometry | where the plasma may be | geometric |

$$\text{measurements} + \text{actuator currents} + \text{machine model} \rightarrow \text{EFIT inputs}$$

The first figure is what EFIT was given on the representative slice, family by family; the second
is which channels were fitted, disabled or missing.

In [ ]:
vaft.omas.plot_equilibrium_overview_constraints(ods, time_slice=i_rep, figsize=(12, 9))
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_overview_constraint_coverage(ods, time_slice=i_rep, figsize=(12, 6))
plt.show()

#### 4.2 Measurement, uncertainty, weight and residual

For each channel $i$ EFIT stores the measured value $y_i$, the reconstructed value
$y_i^{\mathrm{model}}$, a weight $w_i$ and its contribution to $\chi^2$. The quantity that says
whether the fit explains the channel is the **normalised residual**

$$r_i = \frac{y_i - y_i^{\mathrm{model}}}{\sigma_i}, \qquad \chi^2 = \sum_i r_i^2 ,$$

and it is only as meaningful as $\sigma_i$. The residual view draws $y_i - y_i^{\mathrm{model}}$
per family; `fit_quality_metrics` reduces it to numbers.

In [ ]:
vaft.omas.plot_equilibrium_overview_residuals(ods, time_slice=i_rep, figsize=(12, 8))
plt.show()

In [ ]:
metrics = fit_quality_metrics(ods, time_slice=i_rep)
print(f"slice {i_rep}: total chi-square {metrics['chi_squared_total']:.4g}, "
      f"fitted channels {metrics['fitted_channel_count']}")
print(" family        enabled   chi-square sum   share of total")
for family, share in metrics["chi_squared_share"].items():
    record = metrics["families"].get(family) or metrics["scalars"].get(family) or {}
    chi2 = record.get("chi_squared_sum", record.get("chi_squared", np.nan))
    enabled = record.get("channels", {}).get("enabled", 1)
    print(f" {family:14s} {enabled:7d}   {chi2:14.4g}   {share:14.10f}")
if not metrics["chi_squared_share"]:
    print(" (no family carries EFIT's reconstructed values on this slice)")

# EFIT's reconstructed values and chi-squares come from its m-file; a product whose
# EFIT run wrote none has the measurements and weights but nothing to compare them to.
if "ip" in metrics["scalars"]:
    ip_record = metrics["scalars"]["ip"]
    print(f"Ip: measured {ip_record['measured'] / 1e3:.4f} kA, reconstructed {ip_record['reconstructed'] / 1e3:.4f} kA")
    print(f"Ip chi-square as EFIT reported it  : {ip_record.get('chi_squared_efit_reported', np.nan):.4g}")
    print(f"  of which vessel accounting term  : {ip_record.get('vessel_accounting_term', np.nan):.4g}")
    print(f"Ip chi-square, plasma-only residual: {ip_record['chi_squared']:.3g} "
          f"({ip_record.get('source', 'as stored')})")
    print(f"Ip share of the recomputed chi-square: {metrics['chi_squared_share'].get('ip', np.nan):.2g}")
else:
    print("Ip: no EFIT reconstructed value or chi-square on this slice "
          "(the EFIT run wrote no m-file), so there is no Ip residual to read.")

Read those numbers slowly, because they are not what a textbook fit looks like.

- **EFIT reports an $I_p$ $\chi^2$ of about 200, and all of it is vessel accounting.** EFIT's
  current constraint assumes a Rogowski coil that encloses the vessel, so it adds the summed vessel
  current to the plasma current before comparing. VEST's inner Rogowski links no vessel current, so
  that term is not a misfit ([#918](https://github.com/VEST-Tokamak/vaft/issues/918)). VAFT
  recomputes the plasma-only residual $(I_p^{\mathrm{meas}} - I_p^{\mathrm{rec}})^2/\sigma^2$, and
  it is essentially zero: the reconstructed $I_p$ equals the measured one to float32 precision.
- **What remains is dominated by the probes, and it is tiny** — $10^{-9}$ in total — because of
  the uncertainties EFIT was actually given. The next cell compares the $\sigma$ the diagnostics recorded
  (`measured_error_upper`) with the $\sigma$ EFIT fitted against, recovered from its own
  $\chi^2$ as $\sigma_\mathrm{eff} = k / w$.

In [ ]:
print(" family        stored sigma (median)   sigma EFIT used (median)   ratio")
for family in ("bpol_probe", "flux_loop"):
    table = constraint_table(ods, time_slice=i_rep, family=family)
    fitted = table.mask("enabled")
    if not fitted.any():
        print(f" {family:14s} no fitted channels")
        continue
    k, _spread = sigma_unit_factor(table)
    if not np.isfinite(k):
        # k is recovered from EFIT's own chi-square; without the m-file there is none
        print(f" {family:14s} {np.nanmedian(table.uncertainty[fitted]):20.3g}   "
              f"{'no EFIT chi-square':>24s}")
        continue
    sigma_eff = k / table.weight[fitted]
    ratio = np.nanmedian(sigma_eff / table.uncertainty[fitted])
    print(f" {family:14s} {np.nanmedian(table.uncertainty[fitted]):20.3g}   "
          f"{np.nanmedian(sigma_eff):24.3g}   {ratio:8.2g}")

diamagnetic = constraint_table(ods, time_slice=i_rep, family="diamagnetic_flux", is_array=False)
if not len(diamagnetic.measured):
    print("diamagnetic flux: not an input on this slice")
elif not (np.isfinite(diamagnetic.reconstructed[0]) and np.isfinite(diamagnetic.chi_squared[0])):
    print(f"diamagnetic flux: measured {diamagnetic.measured[0] * 1e3:+.3f} mWb, "
          "no EFIT reconstructed value or chi-square on this slice")
else:
    measured, reconstructed = diamagnetic.measured[0], diamagnetic.reconstructed[0]
    chi2 = diamagnetic.chi_squared[0]
    # EFIT compared its reconstruction with the |flux| it was given, not the signed one
    sigma_used = abs(abs(measured) - reconstructed) / np.sqrt(chi2) if chi2 > 0 else np.nan
    print(f"diamagnetic flux: measured {measured * 1e3:+.3f} mWb, "
          f"reconstructed {reconstructed * 1e3:+.3f} mWb, chi-square {chi2:.2g}")
    print(f"  stored sigma {diamagnetic.uncertainty[0] * 1e3:.3g} mWb, stored weight {diamagnetic.weight[0]:g}; "
          f"sigma EFIT used |(|measured| - reconstructed)| / sqrt(chi-square) = {sigma_used:.3g} Wb")

The honest reading of this EFIT fit:

- EFIT's $\sigma$ is its weight times $10^4$: a probe is given about **1000 T**, a flux loop
  100 Wb/rad (the table prints it in Wb, $2\pi$ larger — the reconstructed flux-loop values carry
  the same $2\pi$ so that they match the measured webers), $I_p$ 10 kA, and the PF currents sit at
  EFIT's 0.1 % `SERROR` floor. The diagnostics recorded uncertainties of a few percent. So
  $\sigma_\mathrm{eff}/\sigma$ is of order $10^6$: **the weights, not the diagnostics, set this
  fit**, and a normalised residual of $10^{-5}$ says nothing about agreement.
- The **diamagnetic loop** follows the same rule. The archived k-file submitted $|\Delta\Phi|$ —
  `DFLUX` positive, while the stored measurement is negative — with `SIGDLC` $= 10^{7}$ mWb, i.e.
  the stored weight 1 times $10^4$ Wb; the last printed line recovers that $\sigma$ from EFIT's own
  $\chi^2$. Against a $\sigma$ of $10^4$ Wb a mismatch of a fraction of a mWb (reconstruction
  positive, measurement negative) costs a $\chi^2$ of order $10^{-16}$ or less: it did not constrain
  the fit, so neither did the plasma pressure.

None of this makes the flux map wrong. It means the stored $\chi^2$ cannot be used as evidence of
fit quality, and the next view makes that visible channel by channel: stored $\sigma$ against
$\sigma_\mathrm{eff}$, the weight, and the normalised residual against the $\pm 2\sigma$ band.

In [ ]:
vaft.omas.plot_equilibrium_overview_constraint_weights(ods, time_slice=i_rep, figsize=(15, 10))
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_overview_fit_quality(ods, time_slice=i_rep, figsize=(15, 9))
plt.show()

#### 4.3 The kinetic diagnostics of VEST

| diagnostic | measures | role | in VAFT today |
| --- | --- | --- | --- |
| Thomson scattering | $T_e(R), n_e(R)$ | local electron-profile constraint | fitted (`profile_fitting_thomson_scattering`) and used as the kinetic-EFIT pressure constraint |
| ion Doppler / CES | $T_i(R), v_\phi(R)$ of C$^{3+}$ | local ion-profile constraint | fitted (`profile_fitting_charge_exchange`) |
| interferometer | $\int n_e\,dl$ | line-integrated density | mapped (`interferometer` IDS), not yet a fit constraint |
| two-filter SXR | $T_e$ proxy | auxiliary electron temperature | computed (`vaft.process.soft_x_rays`), not yet a fit constraint |
| diamagnetic flux | pressure integral | total-pressure consistency | an EFIT input (see 4.2 for how it was weighted) |
| spectroscopy | impurity lines | composition, radiation | line intensities only (session 02) |
| visible bremsstrahlung | $Z_\mathrm{eff}$ information | composition, resistivity | not implemented — see 4.10 |

#### 4.4 Local, integral and proxy constraints

They are not the same kind of information, and they need different fitting semantics.

- **Local point measurements** — Thomson gives $T_e(R_i), n_e(R_i)$ at each scattering volume.
  One number, one place.
- **Line-integrated measurements** — an interferometer gives $N_L = \int_\mathrm{LOS} n_e\,dl$.
  A profile is constrained only through a forward model of the chord.
- **Model-dependent proxies** — two-filter SXR $T_e$ depends on the emissivity spectrum and the
  filter response it assumes; it is a model output treated as a measurement.

#### 4.5 Mapping measurements into flux coordinates

A profile point is made by mapping a diagnostic's position through an equilibrium:

$$(R_i, Z_i) \rightarrow \psi_{N,i} \rightarrow \rho_{\mathrm{tor},N,i} .$$

First, where the channels are: Thomson and charge-exchange positions on the kinetic EFIT's flux map.

In [ ]:
ts_r = np.asarray(kinetic["thomson_scattering.channel.:.position.r"], dtype=float)
ts_z = np.asarray(kinetic["thomson_scattering.channel.:.position.z"], dtype=float)
cx_r = np.array([np.ravel(kinetic[f"charge_exchange.channel.{c}.position.r.data"])[0]
                 for c in range(len(kinetic["charge_exchange.channel"]))])
cx_z = np.array([np.ravel(kinetic[f"charge_exchange.channel.{c}.position.z.data"])[0]
                 for c in range(len(kinetic["charge_exchange.channel"]))])

figure, axis = plt.subplots(figsize=(6, 8))
vaft.omas.plot_equilibrium_field_psi(eq_kinetic, ax=axis, overlay=("wall", "boundary", "axis"))
axis.plot(cx_r, cx_z, "s", ms=3, color="tab:orange", label=f"charge exchange ({len(cx_r)})")
axis.plot(ts_r, ts_z, "o", ms=6, color="tab:red", label=f"Thomson ({len(ts_r)})")
axis.legend(loc="upper right", fontsize="small")
plt.show()

Now map the same channels through two reconstructions of the same instant — the magnetic and the
kinetic EFIT. `compare_flux_mapping` returns one record per equilibrium.

In [ ]:
mapped = vaft.process.profile.compare_flux_mapping(
    kinetic, {"magnetic EFIT": eq_magnetic, "kinetic EFIT": eq_kinetic}, diagnostic="thomson_scattering",
    time=t_kinetic,
)
rho_magnetic = mapped["magnetic EFIT"].select("rho_tor_norm")
rho_kinetic = mapped["kinetic EFIT"].select("rho_tor_norm")
print(" channel   R [m]    rho_N magnetic   rho_N kinetic   shift")
for channel, (r, a, b) in enumerate(zip(ts_r, rho_magnetic, rho_kinetic)):
    print(f"{channel:8d}   {r:.3f}   {a:14.3f}   {b:13.3f}   {b - a:+.3f}")
print(f"largest rho_N shift between the two equilibria: {np.nanmax(np.abs(rho_kinetic - rho_magnetic)):.3f}")

**The measured $T_e$ or $n_e$ does not change with the equilibrium; the radial coordinate assigned
to it does.** A channel outside an equilibrium's LCFS maps to NaN and cannot enter a profile fit
at all — that is why a Thomson channel near the edge can appear or disappear between two
reconstructions.

#### 4.6 Kinetic profile fitting

A fit turns points with uncertainties into a function of the flux coordinate,

$$\{(\rho_i, T_{e,i}, \sigma_{T_e,i})\} \rightarrow T_e(\rho), \qquad \{(\rho_i, n_{e,i}, \sigma_{n_e,i})\} \rightarrow n_e(\rho),$$

and the same for $T_i(\rho)$ and $v_\phi(\rho)$ from charge exchange. The fit views draw measured
channels with their error bars, the channels the fitter refused (hollow, with the reason in the
report), the fitted curve, the coordinate and the method. Here is $T_e$ fitted twice — through
the magnetic and through the kinetic EFIT.

In [ ]:
vaft.omas.plot_thomson_scattering_profile_fit(
    kinetic, field="te", coordinate="rho_tor_norm",
    equilibrium={"magnetic EFIT": eq_magnetic, "kinetic EFIT": eq_kinetic},
)
plt.show()

In [ ]:
vaft.omas.plot_thomson_scattering_profile_fit(kinetic, field="ne", coordinate="r_major")
plt.show()

#### 4.7 Statistical fit quality

A curve through points is not evidence. The reports behind those views state it in numbers:

$$r_i = \frac{y_i - f(x_i)}{\sigma_i}, \qquad \chi^2 = \sum_i r_i^2, \qquad \chi^2_\nu = \frac{\chi^2}{N - k},$$

with $N$ the channels used and $k$ the parameter count — the polynomial's coefficients, or for a
Gaussian process the effective number $\mathrm{tr}(S)$ of its smoother. $\chi^2_\nu \approx 1$
means the fit explains the points to within their stated uncertainties; $\gg 1$ means it does not
(or the $\sigma$ are too small); $\ll 1$ means the $\sigma$ are generous or the model has too much
freedom. That is why it, and not $R^2$, is the primary metric.

In [ ]:
t_kinetic_ms = t_kinetic * 1e3
# time= picks the equilibrium slice nearest the fit time; eq_kinetic has one slice, a
# database ODS has many, and slice 0 is then not the instant being fitted
ts_positions = vaft.process.profile.equilibrium_mapping_thomson_scattering(kinetic, eq_kinetic, time=t_kinetic)
cx_positions = vaft.process.profile.equilibrium_mapping_charge_exchange(kinetic, eq_kinetic, time=t_kinetic)

ts_reports = vaft.process.profile.profile_fit_report_thomson_scattering(
    kinetic, t_kinetic_ms, ts_positions, coordinate="rho_tor_norm")
ts_gp = vaft.process.profile.profile_fit_report_thomson_scattering(
    kinetic, t_kinetic_ms, ts_positions, coordinate="rho_tor_norm",
    fitting_function_te="gp", fitting_function_ne="gp")
cx_reports = vaft.process.profile.profile_fit_report_charge_exchange(
    kinetic, t_kinetic_ms, cx_positions, coordinate="rho_tor_norm")
cx_free = vaft.process.profile.profile_fit_report_charge_exchange(
    kinetic, t_kinetic_ms, cx_positions, coordinate="rho_tor_norm", fitting_function_vtor="free_polynomial")

for report in (*ts_reports.values(), ts_gp["t_e"], *cx_reports.values(), cx_free["velocity_tor"]):
    print(report.summary())
te_report = ts_reports["t_e"]
print("T_e normalised residuals of the used channels:", np.round(te_report.normalized_residual[te_report.used], 2))
print("GP parameter count:", ts_gp["t_e"].parameter_count_definition)

Four things in those lines are worth a sentence each.

- **Thomson has five usable points.** Two of the seven channels are refused (the report says
  which, and why), so a cubic leaves $\nu = 2$ degrees of freedom. A $\chi^2_\nu$ from two degrees
  of freedom is a weak statement either way.
- **The Gaussian-process $T_e$ fit collapses to a constant** ($k \approx 1$): with five points the
  hyperparameter optimiser prefers a length scale longer than the plasma, and the smoother becomes
  the sample mean. That is not a GP failure; it is what five points can support.
- **$T_i$ has two branches near the axis.** The charge-exchange chord runs through the magnetic
  axis, so channels just inboard and just outboard of it map to the same $\rho$ — and they do not
  agree: $T_i$ keeps rising inward *past* the axis the equilibrium puts at $R \approx 0.43$ m. If
  $T_i$ is a flux function, the ion profile's peak and the reconstructed axis are not at the same
  place; a single curve in $\rho$ cannot follow both branches, and most of $\chi^2_\nu$ is them.
- **$v_\phi$ with the default polynomial is pinned at the edge** — a boundary condition that
  suits a temperature and not a rotation — and its $\chi^2_\nu$ is near a hundred. Freeing the
  edge (`free_polynomial`) lowers it about six-fold. A large $\chi^2_\nu$ is a question about
  the *model* (and here the mapping), not only about the data.

The same four fits, drawn:

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(20, 4.5))
vaft.omas.plot_thomson_scattering_profile_fit(kinetic, ax=axes[0], field="te", coordinate="rho_tor_norm",
                                              fitting_function="gp")
vaft.omas.plot_charge_exchange_profile_fit(kinetic, ax=axes[1], field="ti", coordinate="rho_tor_norm")
vaft.omas.plot_charge_exchange_profile_fit(kinetic, ax=axes[2], field="vphi", coordinate="rho_tor_norm")
vaft.omas.plot_charge_exchange_profile_fit(kinetic, ax=axes[3], field="vphi", coordinate="rho_tor_norm",
                                           fitting_function="free_polynomial")
figure.tight_layout()
plt.show()

#### 4.8 Weights, priors and constraints

Every reconstruction is, generically,

$$\min_\theta \sum_i w_i \left[y_i - f(x_i;\theta)\right]^2 + \lambda R(\theta), \qquad w_i \sim \frac{1}{\sigma_i^2}\ \text{by default},$$

but in practice $w_i$ also encodes channel validity, diagnostic reliability, correlations and
radial coverage, and $R(\theta)$ encodes smoothness, positivity and axis/edge boundary conditions —
the physicality guard that refused a cubic $n_e$ above is one such prior. Kinetic EFIT exposes the
same two knobs for its pressure constraint as `SIGPRE` (uncertainty) and `FWTPRE` (weight); a weight
only matters relative to the constraints it competes with.

#### 4.9 From primitive profiles to the plasma state

The fitted primitives are $n_e, T_e, T_i, v_\phi$. Pressure is *derived*:

$$p_e = n_e k_B T_e, \qquad p_i = \sum_s n_s k_B T_s, \qquad p_\mathrm{th} = p_e + p_i .$$

The `core_profiles` of 48224 store $n_e$, $T_e$, and one ion species (H$^+$) with $T_i$ from the
C$^{3+}$ charge-exchange fit and $n_i = n_e$ — two assumptions, stated: the main ions have the
impurity's temperature, and the plasma is pure hydrogen.

In [ ]:
core = kinetic["core_profiles.profiles_1d.0"]
rho = np.asarray(core["grid.rho_tor_norm"], dtype=float)
n_e = np.asarray(core["electrons.density_thermal"], dtype=float)
t_e = np.asarray(core["electrons.temperature"], dtype=float)
n_i = np.asarray(core["ion.0.density_thermal"], dtype=float)
t_i = np.asarray(core["ion.0.temperature"], dtype=float)

p_e = vaft.formula.electron_pressure(n_e, t_e)
p_i = vaft.formula.ion_pressure(n_i, t_i)
p_thermal = p_e + p_i

# Each g-file is one instant; find its slice by time, and derive rho_tor_norm, shape, beta and li.
for case in equilibria.values():
    vaft.omas.update_equilibrium_derived_profiles(case)
i_eq = int(np.argmin(np.abs(np.asarray(eq_kinetic["equilibrium.time"], dtype=float) - t_kinetic)))
efit_profiles = eq_kinetic[f"equilibrium.time_slice.{i_eq}.profiles_1d"]
efit_rho = np.asarray(efit_profiles["rho_tor_norm"], dtype=float)
efit_pressure = np.asarray(efit_profiles["pressure"], dtype=float)

figure, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(rho, p_e, label=r"$p_e = n_e T_e$")
axis.plot(rho, p_i, label=r"$p_i = n_i T_i$ ($n_i = n_e$, H$^+$)")
axis.plot(rho, p_thermal, "k", label=r"$p_\mathrm{th} = p_e + p_i$")
axis.plot(rho, np.asarray(core["pressure_thermal"], dtype=float), "k:", label="core_profiles.pressure_thermal (stored)")
axis.plot(efit_rho, efit_pressure, "r--", label=r"kinetic EFIT $p(\psi)$")
axis.set_xlabel(r"$\rho_{\mathrm{tor},N}$")
axis.set_ylabel("pressure [Pa]")
axis.set_title(f"#{KINETIC_SHOT}: derived pressure at {t_kinetic * 1e3:.0f} ms")
axis.legend(fontsize="small")
axis.grid(alpha=0.5)
plt.show()

print(f"on axis: p_e = {p_e[0]:.0f} Pa, p_i = {p_i[0]:.0f} Pa, p_th = {p_thermal[0]:.0f} Pa, "
      f"kinetic EFIT p = {efit_pressure[0]:.0f} Pa")
print(f"T_i / T_e on axis = {t_i[0] / t_e[0]:.2f}")

Three readings. The electron pressure dominates — $T_i/T_e$ is printed, and with $n_i = n_e$ the
ratio of the pressures is the ratio of the temperatures. The black curve lies on the stored
`pressure_thermal`, so `core_profiles` used the same law. And the kinetic EFIT pressure follows
$p_\mathrm{th}$ because it was *constrained* by these points; where Thomson has no channel (the
edge) the EFIT curve is its own source functions' shape, not a measurement.

#### 4.10 Effective charge $Z_\mathrm{eff}$

$$Z_\mathrm{eff} = \frac{\sum_s n_s Z_s^2}{n_e}, \qquad n_e = \sum_s Z_s n_s \quad \text{(quasi-neutrality)} .$$

$Z_\mathrm{eff}$ is not another local profile point like a Thomson channel. It is additional
information about *composition*, and it enters everything downstream: the ion density (and so
$p_i$), Spitzer resistivity and therefore the current profile, collisionality, and radiation. For
one fully stripped impurity in a hydrogen plasma the two relations above invert:

$$\frac{n_I}{n_e} = \frac{Z_\mathrm{eff} - 1}{Z_I (Z_I - 1)} .$$

In [ ]:
carbon_fraction = 0.01                               # n_C / n_e, an assumption for illustration
n_hydrogen = n_e * (1 - 6 * carbon_fraction)         # quasi-neutrality with C6+
n_carbon = n_e * carbon_fraction
plasma = n_e > 0                                     # Z_eff is undefined where n_e = 0
z_eff = vaft.formula.z_eff_from_n_s_Z_s(np.stack([n_hydrogen[plasma], n_carbon[plasma]], axis=-1), [1, 6],
                                        n_e[plasma])        # species on the last axis
recovered = vaft.formula.impurity_fraction_from_effective_charge(float(z_eff[0]), 6)
print(f"1% carbon: Z_eff = {float(z_eff[0]):.2f}; inverted back: n_C/n_e = {recovered:.4f}")

p_i_with_carbon = vaft.formula.ion_pressure(n_hydrogen, t_i) + vaft.formula.ion_pressure(n_carbon, t_i)
print(f"on axis, p_i with that carbon is {1 - p_i_with_carbon[0] / p_i[0]:.0%} below the pure-hydrogen p_i")
for z in (1.5, 2.0, 3.0):
    print(f"Z_eff = {z}: a carbon-only explanation needs n_C/n_e = "
          f"{vaft.formula.impurity_fraction_from_effective_charge(z, 6):.4f}")

VAFT has the composition algebra — `z_eff_from_n_s_Z_s`, the single-impurity inversion, and the
charge-state balance in `vaft.process.atomic` — but **no visible-bremsstrahlung-to-$Z_\mathrm{eff}$
profile workflow yet**. On VEST today $Z_\mathrm{eff}$ is an assumption you choose and state, not
a reconstructed profile.

#### 4.11 Measured, reconstructed, derived

```text
Measured                TS Te ± σ,   CX Ti ± σ,   probes, flux loops, Ip
   ↓
Fitted / reconstructed  Te(ρ), ne(ρ), Ti(ρ), vφ(ρ);   ψ(R,Z), p'(ψ), FF'(ψ)
   ↓
Derived                 pe, pi, p_th, a/L_Te, ν*, ρ*, β, li, q95, κ, δ
```

| layer | IDS | example |
| --- | --- | --- |
| measured | `thomson_scattering`, `charge_exchange`, `magnetics`, `pf_active` | `thomson_scattering.channel[:].t_e.data` |
| fitted / reconstructed | `core_profiles`, `equilibrium` (and `equilibrium.time_slice[:].constraints`, the bridge between them) | `core_profiles.profiles_1d[:].electrons.temperature` |
| derived | fields VAFT adds to `equilibrium` and `core_profiles`, or computes on demand with `vaft.formula` | `global_quantities.beta_normal`, $p_e$ above |

Keep the three apart when you quote a number: say which layer it came from.

### Step 5 — the equilibrium EFIT reconstructed for `SHOT`

#### What an EFIT file does *not* carry

Ask `SHOT` for its normalised beta or its internal inductance and VAFT refuses, because a g-file
stores only `psi`, `p`, `p'`, `F`, `FF'` and `q`. Volume, flux-surface averages, the shape,
beta and `li` have to be *derived* by tracing the flux surfaces. That derivation is one call.

In [ ]:
names = {row["name"] for row in vaft.omas.available_plots(ods)}
print("beta_n available? ", "equilibrium_time_beta_n" in names)
print("li available?     ", "equilibrium_time_li" in names)
print("profiles_1d fields:", len(ods[f"equilibrium.time_slice.{i_rep}.profiles_1d"]))

In [ ]:
vaft.omas.update_equilibrium_derived_profiles(ods)

names = {row["name"] for row in vaft.omas.available_plots(ods)}
print("beta_n available? ", "equilibrium_time_beta_n" in names)
print("li available?     ", "equilibrium_time_li" in names)
print("profiles_1d fields:", len(ods[f"equilibrium.time_slice.{i_rep}.profiles_1d"]))

Seven fields became twenty-four, and plots that did not exist a moment ago now do. **A plot VAFT
refuses is often a quantity nobody has derived yet, not a quantity the shot lacks.** Read the
messages too: the degenerate slice is skipped because it has nothing to derive, and the stored
vacuum field `b0 * r0` disagrees with the TF product, so VAFT takes $R_0$ from `tf` — the
database's `b0` is known to drift within a shot
([#325](https://github.com/VEST-Tokamak/vaft/issues/325)).

Two cautions about what you just computed:

- **`li` here is `li(3)`**, the ITER/Jackson definition $2\int B_p^2\,dV / (\mu_0^2 I_p^2 R_0)$.
  VAFT does not implement `li(1)` or `li(2)`, and they are not interchangeable.
- **"beta poloidal" names three different quantities** in common use, differing by up to 26% on
  VEST. VAFT stores the Data Dictionary one. Always say which you mean.

#### Temporal evolution

Read the histories as motion and shape, not as time series: the current decays, the axis moves,
the plasma shrinks and changes shape, and $q_{95}$, $\beta_p$ and $\ell_i$ follow.

In [ ]:
vaft.omas.plot_equilibrium_overview_histories(ods)
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_time_shape(ods)
plt.show()

shape_rows = []
for i in usable_slices:
    boundary = ods[f"equilibrium.time_slice.{i}.boundary"]
    quantities = ods[f"equilibrium.time_slice.{i}.global_quantities"]
    shape_rows.append((times[i], ip[i], float(boundary["minor_radius"]), float(boundary["elongation"]),
                       float(boundary["triangularity"]), float(quantities["q_95"]),
                       float(quantities["beta_normal"]), float(quantities["li_3"])))
print(" t [ms]   Ip [kA]   a [m]   kappa   delta   q95     beta_N   li_3")
for t, current, a, kappa, delta, q95, beta_n, li in shape_rows:
    print(f"{t * 1e3:6.1f}   {current / 1e3:7.1f}   {a:.3f}   {kappa:5.2f}   {delta:5.2f}   "
          f"{q95:5.2f}   {beta_n:6.3f}   {li:.3f}")

#### The representative slice

The flux map at the representative slice and at the last usable one, side by side — how far the
plasma moved and shrank — with the topology classifier of step 2 applied to a real discharge.

In [ ]:
i_last = int(usable_slices.max())
figure, axes = plt.subplots(1, 2, figsize=(10, 7))
for axis, i in zip(axes, (i_rep, i_last)):
    vaft.omas.plot_equilibrium_field_psi(ods, time_slice=i, ax=axis,
                                         overlay=("wall", "boundary", "axis", "x_points"))
    axis.set_title(f"#{SHOT} at {times[i] * 1e3:.1f} ms")
    representation = vaft.process.equilibrium.derive_boundary_representation(
        vaft.process.equilibrium.as_equilibrium(ods, time_index=i))
    saddles = representation.x_points
    on_boundary = [point for point in saddles if point.active]
    print(f"slice {i} ({times[i] * 1e3:.1f} ms): topology {representation.topology.value}; "
          f"{len(saddles)} saddles of psi on the grid, {len(on_boundary)} on the boundary")
    if saddles:
        nearest = saddles[int(np.argmin([abs(point.psi_n - 1) for point in saddles]))]
        print(f"  saddle nearest the boundary flux: R = {nearest.r:.3f} m, Z = {nearest.z:+.3f} m, "
              f"psi_N = {nearest.psi_n:.3f}")
figure.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=i_rep, ax=axes[0])
vaft.omas.plot_equilibrium_profile_pressure(ods, time_slice=i_rep, ax=axes[1])
vaft.omas.plot_equilibrium_geometry_boundary(ods, time_slice=i_rep, ax=axes[2])
figure.tight_layout()
plt.show()

#### The reconstruction on the camera

Session 02 showed the camera frames; here they are a **validation** of the reconstruction. The
LCFS is projected through the calibrated camera model onto the frame nearest a slice's time: if
the light and the boundary disagree, one of them is wrong. Frames are images, not an IDS, so two
mapper calls write them in — only the two frames nearest the representative and the last usable
slice, because a frame costs about 10 MB in an ODS.

In [ ]:
import cv2
from vaft.machine_mapping.camera_visible import vfit_camera_visible_dynamic, vfit_camera_visible_static

camera_frames = []
try:
    camera_frames = vaft.data.sample_camera_visible_frame_paths(SHOT)
except FileNotFoundError as error:
    print(f"no packaged camera frames for shot {SHOT}: {error}")

camera_times = []
if camera_frames and "camera_visible" not in ods:
    frame_times = np.array([time_s for time_s, _ in camera_frames])
    wanted = [times[i_rep], times[usable_slices.max()]]
    picked = [int(np.argmin(np.abs(frame_times - t))) for t in wanted]
    images = [cv2.imread(str(camera_frames[k][1]), cv2.IMREAD_GRAYSCALE) for k in picked]
    vfit_camera_visible_static(ods, lines_n=images[0].shape[0], columns_n=images[0].shape[1],
                               channel_name="Fast Camera")
    vfit_camera_visible_dynamic(ods, images=images, times_s=[frame_times[k] for k in picked])
    camera_times = [float(frame_times[k]) for k in picked]
    print("frames at " + ", ".join(f"{t * 1e3:.1f} ms" for t in camera_times)
          + " for slices at " + ", ".join(f"{t * 1e3:.1f} ms" for t in wanted))

In [ ]:
if camera_times:
    figure, axes = plt.subplots(1, len(camera_times), figsize=(7 * len(camera_times), 5))
    for axis, when in zip(np.atleast_1d(axes), camera_times):
        vaft.omas.plot_camera_visible_image_efit_overlay(ods, shot=SHOT, time=when, ax=axis,
                                                         theta_deg_range=(-8.0, 8.0))
    figure.tight_layout()
    plt.show()
else:
    print(f"shot {SHOT} has no camera frames: the EFIT overlay cannot be checked against the light")

`theta_deg_range` matters: every overlay is swept toroidally, so at the default sweep the flux
surfaces project into a filled sheet, and a narrow sweep is what makes them read as contours. Two
frames, two slices: the boundary should shrink between them the way the current and the minor
radius in the table above say it does.

### Step 6 — flux coordinates and 1-D profiles

The same `q` profile, drawn against three coordinates. After `update_equilibrium_derived_profiles`
the `rho_tor_norm` is the real toroidal-flux radius.

In [ ]:
profiles = ods[f"equilibrium.time_slice.{i_rep}.profiles_1d"]
rho_derived = np.asarray(profiles["rho_tor_norm"], dtype=float)
gap = float(np.nanmax(np.abs(rho_derived - np.sqrt(psi_n))))
where = float(np.sqrt(psi_n)[np.nanargmax(np.abs(rho_derived - np.sqrt(psi_n)))])
print(f"derived rho_tor_norm vs sqrt(psi_N): max difference {gap:.3f}, at sqrt(psi_N) = {where:.2f}")

In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=i_rep, coordinate="psi_norm")
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=i_rep, coordinate="rho_tor_norm")
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_profile_q(ods, time_slice=i_rep, coordinate="r_major")
plt.show()

**Radial coordinate choice is part of the physical statement, not a plotting option.** The curve
is the same physics; the regions that stretch and shrink between $\psi_N$ and $\rho_{\mathrm{tor},N}$
are where $q$ departs from its average — the edge, where $q$ climbs, is compressed in $\psi_N$.
A gradient, a pedestal width or a "core" value means different things on each ruler, so state it.
VAFT will not draw a curve against an abscissa it cannot honestly name, and falls back to a
coarser coordinate rather than mislabel one.

### Step 7 — the kinetic state of 48224

With the profiles reconstructed on an equilibrium, the plasma state is an analysis object.
First the fitted electron profiles stored in `core_profiles`:

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
vaft.omas.plot_electron_temperature_profile(kinetic, ax=axes[0])
vaft.omas.plot_electron_density_profile(kinetic, ax=axes[1])
figure.tight_layout()
plt.show()

Then the derived layer: the pressure components of 4.9 against the equilibrium's own pressure,
normalised gradients, and the dimensionless parameters.

$$\frac{a}{L_{T_e}} = -\frac{a}{T_e}\frac{dT_e}{dr}, \qquad
\nu_e^* = 6.921\times10^{-18}\,\frac{q R\, n_e Z_\mathrm{eff} \ln\Lambda_e}{\epsilon^{3/2} T_e^2}, \qquad
\rho_* = \frac{\rho_i}{a} .$$

On $\rho_{\mathrm{tor},N}$ `normalized_gradient_scale_length(rho, y, 1)` returns $-d\ln y/d\rho$,
which is $a/L_y$ only up to the Jacobian $d\rho/dr$ — close to one here, but say so.

In [ ]:
kinetic_root = f"equilibrium.time_slice.{i_eq}"
a_minor = float(eq_kinetic[f"{kinetic_root}.boundary.minor_radius"])
r0 = float(eq_kinetic["equilibrium.vacuum_toroidal_field.r0"])
b0 = abs(float(np.ravel(eq_kinetic["equilibrium.vacuum_toroidal_field.b0"])[0]))
q_on_rho = np.interp(rho, efit_rho, np.abs(np.asarray(efit_profiles["q"], dtype=float)))

core_region = (rho > 0) & (rho < 0.9) & (t_e > 0) & (n_e > 0) & (t_i > 0)   # the edge, where T -> 0, is excluded
x = rho[core_region]
epsilon = x * a_minor / r0                      # r/R with r ~ rho_tor_N * a: an approximation
a_over_lte = vaft.formula.normalized_gradient_scale_length(x, t_e[core_region], 1.0)
a_over_lne = vaft.formula.normalized_gradient_scale_length(x, n_e[core_region], 1.0)
nu_star = {z: vaft.formula.electron_collisionality_sauter(n_e[core_region], t_e[core_region], q_on_rho[core_region],
                                                          r0, epsilon, z) for z in (1.0, 2.0)}
rho_star = vaft.formula.rho_star_from_M_T_B_R_epsilon(1.0, t_i[core_region], b0, r0, a_minor / r0)

figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(x, a_over_lte, label=r"$-d\ln T_e/d\rho$")
axes[0].plot(x, a_over_lne, label=r"$-d\ln n_e/d\rho$")
axes[0].set_ylabel("normalised inverse gradient length")
for z, values in nu_star.items():
    axes[1].semilogy(x, values, label=rf"$\nu_e^*$, $Z_\mathrm{{eff}} = {z:.0f}$")
axes[1].axhline(1.0, color="k", lw=0.8, ls=":")
axes[2].plot(x, rho_star, label=r"$\rho_*$ (H, $T_i$)")
for axis in axes:
    axis.set_xlabel(r"$\rho_{\mathrm{tor},N}$")
    axis.legend(fontsize="small")
    axis.grid(alpha=0.5)
figure.tight_layout()
plt.show()

half = int(np.argmin(np.abs(x - 0.5)))
print(f"at rho_N = {x[half]:.2f}: -dlnTe/drho = {a_over_lte[half]:.2f}, -dlnne/drho = {a_over_lne[half]:.2f}, "
      f"nu_e* = {nu_star[1.0][half]:.2g} (Z_eff 1) / {nu_star[2.0][half]:.2g} (Z_eff 2), rho* = {rho_star[half]:.3f}")
print(f"a = {a_minor:.3f} m, R0 = {r0:.2f} m, B0 = {b0:.3f} T, q at rho_N 0.5 = {q_on_rho[core_region][half]:.2f}")

Where the kinetic pressure entered, the equilibrium changed. The magnetic and kinetic EFIT of the
same instant, on the same coordinate:

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
vaft.omas.plot_equilibrium_profile_q([eq_magnetic, eq_kinetic], label=["magnetic EFIT", "kinetic EFIT"],
                                     coordinate="rho_tor_norm", ax=axes[0])
vaft.omas.plot_equilibrium_profile_pressure([eq_magnetic, eq_kinetic], label=["magnetic EFIT", "kinetic EFIT"],
                                            coordinate="rho_tor_norm", ax=axes[1])
axes[1].plot(rho, p_thermal, "k:", label="p_e + p_i from core_profiles")
axes[1].legend(fontsize="small")
figure.tight_layout()
plt.show()

print(" equilibrium      beta_p    li_3    beta_N    q95    p on axis [Pa]")
for name, case in (("magnetic EFIT", eq_magnetic), ("kinetic EFIT", eq_kinetic), ("CHEASE", eq_chease)):
    i_case = int(np.argmin(np.abs(np.asarray(case["equilibrium.time"], dtype=float) - t_kinetic)))
    quantities = case[f"equilibrium.time_slice.{i_case}.global_quantities"]
    axis_pressure = float(np.asarray(case[f"equilibrium.time_slice.{i_case}.profiles_1d.pressure"])[0])
    print(f" {name:15s} {float(quantities['beta_pol']):7.4f}   {float(quantities['li_3']):5.3f}   "
          f"{float(quantities['beta_normal']):6.3f}   {float(quantities['q_95']):5.2f}   {axis_pressure:10.0f}")

The two reconstructions agree on $q$ and on $\ell_i$ — the magnetics fix the current profile and
the boundary well — and disagree on the pressure: adding the kinetic points roughly doubles
$\beta_p$ and $\beta_N$ and lifts the axis pressure onto $p_e + p_i$. That is the general lesson
of step 4: on VEST, without an internal pressure measurement, the magnetics leave the pressure
nearly free, and whatever $\beta$ a magnetics-only fit reports is largely its own assumption.

### Step 8 — did the reconstruction solve its own equation?

A converged reconstruction is not automatically a *consistent* one. EFIT reports how little
`psi` moved on its last iteration, but that says nothing about whether the flux map it settled on
satisfies the Grad-Shafranov equation with the `p'` and `FF'` it also reports. Evaluate both sides
and compare — on the kinetic EFIT, and on its CHEASE refinement (513 × 513, fixed boundary).

In [ ]:
residual = vaft.omas.compute_grad_shafranov_residual(eq_kinetic)
print(f"points scored     : {int(residual.mask.sum())}")
print(f"median relative   : {float(np.nanmedian(residual.relative)):.4f}")

Outside the separatrix `psi` is not monotonic — it turns around near the poloidal-field coils and
folds back into the same range as the plasma. Scoring by flux value alone re-admits vacuum grid
points, where `p'` and `FF'` are extrapolation and $\Delta^*\psi$ is reading the *coils*; on this
shot that would be a third of the points, and it inflates the answer by more than a decade. VAFT
masks on the boundary instead. **Ask a question only where its terms are defined.**

In [ ]:
refined_residual = vaft.omas.compute_grad_shafranov_residual(eq_chease)

print(f"EFIT   median relative residual: {float(np.nanmedian(residual.relative)):.4f}")
print(f"CHEASE median relative residual: {float(np.nanmedian(refined_residual.relative)):.4f}")

figure, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].semilogy(residual.psi_norm, residual.relative, label="kinetic EFIT")
axes[0].semilogy(refined_residual.psi_norm, refined_residual.relative, label="CHEASE refined")
axes[0].set_xlabel(r"$\psi_N$")
axes[0].set_ylabel("relative Grad-Shafranov residual")
axes[0].legend()
axes[0].grid(alpha=0.5)
vaft.omas.plot_equilibrium_geometry_boundary([eq_kinetic, eq_chease], label=["kinetic EFIT", "CHEASE"],
                                             ax=axes[1])
figure.tight_layout()
plt.show()

Both files close their own equation to within a few percent, which is the honest headline. The
*shapes* are what carry information. The kinetic EFIT's residual is below $10^{-3}$ near the axis
and grows toward the edge, where its coarse grid and its low-order $p'$ and $FF'$ have the least
freedom. CHEASE's is flat, at a few percent everywhere — and a flat *relative* residual is the
signature of a uniform scale mismatch between the stored flux map and the stored source
functions, not of a noisy solve. CHEASE solved on its own flux-coordinate mesh; what is scored
here is its g-file export, re-read on a rectangular grid, whose header even declares the wrong
COCOS (the sample manifest records it). So this residual tests the *file*: it says the CHEASE
export is consistent to a few percent, not that CHEASE solved its equation worse than EFIT.

Refinement is only meaningful if it did not quietly change the equilibrium, and the right panel
checks that: the two boundaries lie on top of each other.

## Interpretation Checkpoints

Each has a definite answer in what you have already printed or plotted.

1. **`Ip chi-square as EFIT reported it` against `plasma-only residual`.** About 200 against
   essentially zero, for an $I_p$ matched to float precision. What would you have concluded from
   EFIT's own $\chi^2$ alone, and why does the Rogowski's geometry decide which number is right?
2. **The stored-versus-used $\sigma$ table.** The ratio is of order $10^6$. If you reran EFIT with
   $\sigma$ equal to `measured_error_upper`, which family would start to dominate $\chi^2$ — and
   would you expect the flux map to move?
3. **The diamagnetic line.** Measured negative, reconstructed positive, $\chi^2$ of order
   $10^{-16}$ because EFIT's $\sigma$ was $10^4$ Wb. What would a diamagnetic constraint weighted by
   its stored $\sigma$ add, and which quantity in the history plot would it pin?
4. **`largest rho_N shift`.** Same Thomson channel, two equilibria of one instant. Is the shift
   larger than the channel spacing? What does that mean for a gradient computed from these points?
5. **The four fit summaries.** Which fit would you distrust because $\chi^2_\nu \gg 1$, which
   because $\nu$ is tiny, and which because $k \approx 1$?
6. **`beta_n available?  False` then `True`.** Why does a *derivation* step change what you are
   allowed to plot?
7. **`derived rho_tor_norm vs sqrt(psi_N)`.** Where is the difference largest, and what would a
   transport study get wrong on the proxy?
8. **The EFIT and CHEASE residuals.** Both are small, and CHEASE's is the larger. Why does a flat
   relative residual point at the exported file rather than at the solver, and what would you check
   first?

## Integrated Analysis

### Part II, level 1 — one discharge, several times

The same analysis at the first, the representative, a middle and the last usable slice of `SHOT`. The slices come from `usable_slices`, so the cell works on any shot.

In [ ]:
picks = sorted({int(usable_slices.min()), i_rep, int(usable_slices[len(usable_slices) // 2]),
                int(usable_slices.max())})
figure, axes = plt.subplots(1, 3, figsize=(16, 5))
for i in picks:
    outline = ods[f"equilibrium.time_slice.{i}.boundary.outline"]
    label = f"{times[i] * 1e3:.1f} ms"
    axes[0].plot(outline["r"], outline["z"], label=label)
    vaft.omas.plot_equilibrium_profile_q(ods, time_slice=i, coordinate="rho_tor_norm", ax=axes[1])
    vaft.omas.plot_equilibrium_profile_pressure(ods, time_slice=i, coordinate="rho_tor_norm", ax=axes[2])
axes[0].set_aspect("equal")
axes[0].set_xlabel("R [m]")
axes[0].set_ylabel("Z [m]")
axes[0].set_title(f"#{SHOT}: LCFS")
axes[0].legend(fontsize="small")
for axis in axes[1:]:
    axis.legend([f"{times[i] * 1e3:.1f} ms" for i in picks], fontsize="small")
figure.tight_layout()
plt.show()

### Part II, level 2 — several discharges

Everything in Part I was written without a shot-specific number, so it runs unchanged on a list.
The commented line is the lab-mode path for any list of shots. Each discharge is compared at its
own representative slice — the usable slice nearest its own current peak — on the same
coordinate, $\rho_{\mathrm{tor},N}$ after derivation.

In [ ]:
shots = [39915, 41524, 41672]
ods_list = [vaft.omas.sample_ods(shot) for shot in shots]
# ods_list = vaft.database.load(shots)  # lab mode: any list of shots, needs HSDS credentials

representative = []
for case in ods_list:
    vaft.omas.update_equilibrium_derived_profiles(case)
    case_times = np.asarray(case["equilibrium.time"], dtype=float)
    case_ip = np.full(len(case_times), np.nan)
    for i in range(len(case_times)):
        root = f"equilibrium.time_slice.{i}"
        has_boundary = f"{root}.boundary.outline.r" in case and len(case[f"{root}.boundary.outline.r"]) > 2
        if f"{root}.global_quantities.ip" in case and has_boundary:
            case_ip[i] = float(case[f"{root}.global_quantities.ip"])
    case_ip[case_ip == 0] = np.nan
    representative.append(int(np.nanargmax(np.abs(case_ip))))
print(f"loaded {len(ods_list)} discharges: {shots}; representative slices {representative}")

In [ ]:
columns = ("shot", "t [ms]", "Ip [kA]", "R_ax [m]", "a [m]", "kappa", "delta", "q95", "beta_p", "beta_N", "li_3")
rows = []
for shot, case, i in zip(shots, ods_list, representative):
    boundary = case[f"equilibrium.time_slice.{i}.boundary"]
    quantities = case[f"equilibrium.time_slice.{i}.global_quantities"]
    rows.append([
        f"{shot}", f"{float(case['equilibrium.time'][i]) * 1e3:.1f}", f"{float(quantities['ip']) / 1e3:.1f}",
        f"{float(quantities['magnetic_axis.r']):.3f}", f"{float(boundary['minor_radius']):.3f}",
        f"{float(boundary['elongation']):.2f}", f"{float(boundary['triangularity']):.2f}",
        f"{float(quantities['q_95']):.2f}", f"{float(quantities['beta_pol']):.3f}",
        f"{float(quantities['beta_normal']):.3f}", f"{float(quantities['li_3']):.3f}",
    ])
widths = [max(len(name), *(len(row[k]) for row in rows)) for k, name in enumerate(columns)]
print("  ".join(name.rjust(width) for name, width in zip(columns, widths)))
for row in rows:
    print("  ".join(cell.rjust(width) for cell, width in zip(row, widths)))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 5))
for shot, case, i in zip(shots, ods_list, representative):
    outline = case[f"equilibrium.time_slice.{i}.boundary.outline"]
    axes[0].plot(outline["r"], outline["z"], label=f"#{shot}")
    vaft.omas.plot_equilibrium_profile_q(case, time_slice=i, coordinate="rho_tor_norm", ax=axes[1])
    vaft.omas.plot_equilibrium_profile_pressure(case, time_slice=i, coordinate="rho_tor_norm", ax=axes[2])
axes[0].set_aspect("equal")
axes[0].set_xlabel("R [m]")
axes[0].set_ylabel("Z [m]")
axes[0].set_title("LCFS at each representative slice")
for axis in axes:
    axis.legend([f"#{shot}" for shot in shots], fontsize="small")
    axis.set_title(axis.get_title().split(" #")[0])
figure.tight_layout()
plt.show()

In [ ]:
vaft.omas.plot_equilibrium_time_shape(ods_list)
plt.show()

Read the table and the profiles together, and keep the caveats attached:

- The later shots carry more current, so at similar field their $q_{95}$ is lower; compare their
  $q$ profiles on $\rho_{\mathrm{tor},N}$, not on $\psi_N$, where a different edge $q$ would
  stretch the axis differently.
- $\beta_p$ and $\beta_N$ come from a pressure that none of these magnetics-only fits
  constrained (step 4.2): they are the *reconstruction's* pressure, not a measured one. The
  kinetic EFIT of 48224 is what a measured pressure changes.
- Shape histories are only as good as the usable window. The table uses each shot's own
  representative slice, chosen by the same rule, never by index.

## Independent Exercise

### Reconstruct, question and compare a discharge of your choice

Pick any VEST discharge (lab mode, `vaft.database.load`) — ideally one with Thomson data — and
write a short report that answers:

1. **The usable window.** Which equilibrium slices are usable, by the rule of step 0?
2. **Representative times.** Choose 2–3 slices and say why.
3. **Geometry and global parameters.** Compare $R_0$, $a$, $\kappa$, $\delta$, $q_{95}$, $\beta_p$,
   $\beta_N$, $\ell_i$ between them.
4. **Coordinates.** Draw one profile on $\psi_N$ and on $\rho_{\mathrm{tor},N}$; where do they differ?
5. **Constraints.** For two families, print measured, reconstructed, residual and $\sigma$; do the
   weights or the diagnostics set the fit?
6. **Kinetic fits.** If Thomson or CES exist, draw raw points against the fitted profile.
7. **Goodness of fit.** Report $\chi^2$, $\nu$ and $\chi^2_\nu$ for each fit.
8. **Derived state.** Compute one pressure or dimensionless quantity, with its assumptions.
9. **Comparison.** Across time, or against another discharge on a common coordinate.
10. **Interpretation.** One paragraph on what physically differs.

In the report, keep five labels apart: what was **measured**, what was **reconstructed**, what was
**assumed**, what was **derived**, and **how well** the reconstruction explains the measurements.
The CHEASE lines in the cell are the forward-model extension: vary the boundary, the pressure and
the current peaking of your equilibrium and watch which derived quantities move. Scaling `FF'`
uniformly does **not** move `li` — CHEASE renormalises the current — so `current_peaking` is the
knob for that.

In [ ]:
# Lab mode: needs HSDS credentials. Replace the shot with the one you chose.
# SHOT = <your shot>
# Swap the lab-mode line in the Load / Prepare Data cell, then rerun Part I from the top:
# usable_slices, i_rep and every step after them follow the new shot.
#
# 5. constraints of one family at the representative slice
# table = constraint_table(ods, time_slice=i_rep, family="bpol_probe")
# print(table.measured, table.reconstructed, table.residual, table.uncertainty, table.weight)
#
# 6-7. kinetic fits and their statistics, when the shot carries Thomson scattering
# if "thomson_scattering" in ods:
#     # map through the equilibrium slice nearest the fit time, paired by time, not by index
#     positions = vaft.process.profile.equilibrium_mapping_thomson_scattering(ods, ods, time=t_rep)
#     reports = vaft.process.profile.profile_fit_report_thomson_scattering(
#         ods, t_rep * 1e3, positions, coordinate="rho_tor_norm")
#     for report in reports.values():
#         print(report.summary())
#
# Forward-model extension: needs CHEASE installed (vaft.code.find_chease_executable()).
# from vaft.code import CHEASEConfig, EquilibriumVariation, scan_chease
# cases = scan_chease(
#     "<a g-file of your equilibrium>",
#     [
#         EquilibriumVariation("control"),
#         EquilibriumVariation("beta_up", pressure_scale=1.5),
#         EquilibriumVariation("li_up", current_peaking=1.0),
#         EquilibriumVariation("elong_up", elongation_scale=1.10),
#     ],
#     config=CHEASEConfig(nideal=6, nw=513, target_psin=0.993, relax=0.5),
#     workdir="outputs/03/scan",
# )
# for case in cases:
#     if case.converged:
#         result = vaft.data.read_geqdsk(case.result.refined_geqdsk).to_omas()
#         vaft.omas.update_equilibrium_derived_profiles(result)
#         print(case.variation.label, vaft.omas.compute_grad_shafranov_residual(result).relative)

## Takeaways and Next Steps

- An equilibrium is a solution of the Grad-Shafranov equation under stated assumptions —
  axisymmetry, no strong flow, isotropic pressure, nested surfaces — and whether a stored one *is*
  such a solution is a question you can ask rather than assume.
- A shape parameterisation (Miller) is not a force-balanced solution (Solov'ev); topology —
  limited, single-null, double-null — is a property of the flux map, decided by its saddles.
- Forward and reconstruction, fixed and free boundary, are two independent axes. EFIT tells you
  what the discharge did; CHEASE what a set of inputs implies, to numerical precision.
- A reconstruction is only as good as its constraints *and their weights*. On these samples the
  weights, not the diagnostics, set the magnetic fit, EFIT's reported $I_p$ $\chi^2$ is vessel
  accounting rather than misfit, and the
  diamagnetic loop did not constrain the pressure. Read the channels, not the headline number.
- The measured value does not depend on the equilibrium; its radial coordinate does. Fit profiles
  in a named coordinate and judge them by $\chi^2_\nu$ with $\nu$ and $k$ stated.
- Measured, reconstructed, assumed, derived: four layers, and a quoted number should say which.
  $Z_\mathrm{eff}$ on VEST is still an assumption.
- A g-file carries six profiles; volume, shape, beta and `li` are derived, and a refused plot is
  often an underived quantity.

**Not covered here.** Straight-field-line coordinates — PEST, Boozer, Hamada — are the natural
next step for anyone heading toward stability analysis, and VAFT does not yet compute them: the
existing helper returns a geometric approximation rather than a flux-coordinate transform. That
gap is tracked as issue #472.

**Next**: Session 04 takes the equilibrium you now trust and looks at what fluctuates on top of
it; session 05 relaxes the assumptions this session made — islands, 3-D fields, and the stability
of the equilibrium itself.